# Phase 2 — Pattern Extraction (color / stripe / spot)

Runs `src/pattern_extractor/` against Phase 1's output (`data/extracted_fish/`) to produce one feature row per image across three independent dimensions: coloring (patternize-derived k-means clustering), spots (blob shape), stripes (region elongation + FFT periodicity).

**No GPU needed.** This is pure NumPy/SciPy/Pillow, deterministic and fast - CLAUDE.md notes this stage deliberately skips resumable per-image state for exactly that reason, a re-run just recomputes everything, cheaply. Use a **CPU runtime** here (`Runtime -> Change runtime type -> CPU`) to save your GPU quota for Phase 1.

**Prerequisite:** Phase 1 must already have produced output in `data/extracted_fish/` - run `Phase1_Fish_Extraction.ipynb` first.

See [README.md](../README.md) (Planned Approach, step 2) for the full citation and design reasoning (the *patternize* reimplementation, the three-dimension split's developmental-biology basis).

## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

## 2. Open the same Drive-resident project Phase 1 used

This must resolve to the **same** `PROJECT_DIR` Phase 1 wrote `data/extracted_fish/` into - Phase 2 reads that output directly via the same relative-path layout. If this is a fresh Colab runtime that never ran Phase 1, this cell clones fresh onto Drive the same way Phase 1's notebook does; if Phase 1 already set it up, this just pulls any code updates.

In [ ]:
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/guptrishi01/Surgeonfish_Neural_Network_Phylogenetics.git"
PROJECT_DIR = Path("/content/drive/MyDrive/Surgeonfish_Neural_Network_Phylogenetics")

if not PROJECT_DIR.exists():
    print(f"Cloning into {PROJECT_DIR} (first time - pulls ~1.8GB, be patient)...")
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(PROJECT_DIR)], check=True)
else:
    print(f"{PROJECT_DIR} already exists - pulling latest code only.")
    subprocess.run(["git", "-C", str(PROJECT_DIR), "pull"], check=True)

In [ ]:
%cd {PROJECT_DIR}

## 3. Add the local package to the path

Base dependencies only (numpy, scipy, Pillow) - already in Colab's default image, no GPU extras needed for this phase.

In [ ]:
import sys
sys.path.insert(0, str(PROJECT_DIR / "src"))

import numpy, scipy, PIL
print("Base deps OK - numpy", numpy.__version__, "scipy", scipy.__version__, "Pillow", PIL.__version__)

## 4. Confirm Phase 1's output is actually there - and check for fully-dropped species

`fish_extractor` only creates a species folder in `data/extracted_fish/` once at least one of that species' images was *accepted* - a species where every image ended up excluded or still flagged has no folder at all, not an empty one, and Phase 2 would silently produce zero rows for it rather than erroring. With 604/1,460 images excluded in the real run, this is worth actually checking rather than assuming it didn't happen - compares the full 64-species list from `data/raw_images/` (untouched by Phase 1, so always complete) against what `data/extracted_fish/` actually has.

In [ ]:
extracted_root = PROJECT_DIR / "data" / "extracted_fish"
raw_root = PROJECT_DIR / "data" / "raw_images"

raw_species = {
    f"{genus.name}/{species.name}"
    for genus in raw_root.iterdir() if genus.is_dir()
    for species in genus.iterdir() if species.is_dir()
}
extracted_species = {
    f"{genus.name}/{species.name}"
    for genus in extracted_root.iterdir() if genus.is_dir()
    for species in genus.iterdir() if species.is_dir()
} if extracted_root.exists() else set()

print(f"{len(raw_species)} species in data/raw_images/, {len(extracted_species)} have at least "
      f"one accepted image in data/extracted_fish/.")

missing = sorted(raw_species - extracted_species)
if missing:
    print(f"\n{len(missing)} species have ZERO accepted images - every image was excluded/flagged:")
    for m in missing:
        print(f"  {m}")
    print("\nThese won't appear in pattern_features.csv at all. Worth deciding now whether that's "
          "expected (e.g. a species with very few source images to begin with) or worth revisiting "
          "in Phase 1 before continuing.")
elif not extracted_species:
    print("Nothing here yet - run Phase1_Fish_Extraction.ipynb first.")
else:
    print("All species have at least one accepted image.")

## 5. Run pattern extraction

Writes one feature row per image (color/spot/stripe columns, plus an `is_reference` column marking the curated seed photo vs. GBIF field images - see README.md's Planned Approach step 3 for why that distinction matters for the aggregation step that comes next in Phase 3) to `reports/pattern_features.csv`.

Logging is configured with `force=True` first so you get a `[n/64] <species>: <count> processed` line per species as it runs - plain `logging.basicConfig()` silently does nothing if the root logger already has handlers configured (easy to hit by accident in a long-lived Colab session that's run other cells before), which would make a perfectly healthy run look just as silent as a stuck one. This stage has no resumable state (a re-run just recomputes everything, see the package docstring), so this progress output is the only visibility available while it's running, not something you can check after the fact. Two real performance fixes went into making this fast enough to watch complete: colour clustering's k-means fit now runs on at most 50,000 randomly-sampled masked-in pixels rather than every single one, and connected-component region-finding (spots/stripes) no longer re-scans the whole image per component - both were previously untested at real (non-resized) crop sizes and could turn a few hundred images into a multi-hour run. If a single species' line takes more than a minute or two to appear now, that's worth flagging back rather than assuming it'll finish.

**If you already have an older `reports/pattern_features.csv`, re-run this step and overwrite it.** The first real run's output was checked against known fish biology (e.g. species with no real stripes at all) and found untrustworthy: 99% of all 856 images were flagged `stripe_present`, regardless of species. Cause: `RegionConfig`'s noise-filtering floor was a fixed 20-pixel absolute count, calibrated against tiny synthetic test images and never exercised at real crop sizes - at real scale it barely filtered anything, so hundreds of small noise/texture regions per image were counted as "stripes." Now scales as a fraction of each image's actual masked area instead of a fixed pixel count - see `RegionConfig.min_region_area_fraction`'s docstring for the full reasoning. This fix changes the actual feature values, not just performance, so old output from before it landed should not be trusted or mixed with new output.

In [ ]:
import logging

logging.basicConfig(
    level=logging.INFO, format="%(asctime)s %(levelname)-8s %(message)s", force=True
)

from pattern_extractor.config import PipelineConfig
from pattern_extractor.pipeline import PatternExtractorPipeline

config = PipelineConfig()  # relative paths, same PROJECT_DIR layout Phase 1 used
rows = PatternExtractorPipeline(config).run()
print(f"Wrote {len(rows)} feature row(s) to {config.output_csv_path.resolve()}")

## 6. Quick sanity check, including real per-species image counts

The README's Phase 3 plan already documents specific sparse-species concerns (e.g. *Naso maculatus* down to a single image) based on Phase 0/1's *pre-review* numbers - this recomputes it from what Phase 1's real review process actually left behind, since exclusions during review could have thinned any species further, not just the ones already flagged as sparse.

In [ ]:
import csv
from collections import Counter

with open(config.output_csv_path, newline="", encoding="utf-8") as f:
    reader = list(csv.DictReader(f))

n_species_seen = len({row["image_key"].split("/")[1] for row in reader})
n_reference = sum(1 for row in reader if row["is_reference"] == "True")
print(f"{len(reader)} image row(s) across {n_species_seen} species; {n_reference} marked is_reference.")

per_species = Counter(row["image_key"].split("/")[1] for row in reader)
sparse = sorted((n, sp) for sp, n in per_species.items() if n <= 5)
print(f"\n{len(sparse)} species with 5 or fewer images after Phase 1's real review:")
for n, sp in sparse:
    print(f"  {sp}: {n}")

reader[:3]

## 6b. Biological sanity check

A summary statistic can look fine while the underlying feature is wrong - this project has already shipped one silently-broken metric before (see the Status section in README.md). This checks the extracted features against a few species whose real pattern is well documented, as a cheap proxy for the full manually-labeled validation split planned but not yet built (README.md's Planned Approach, step 2): a genuinely striped species (*Acanthurus lineatus* - bold blue/black/yellow bands) should score meaningfully higher on `stripe_present`/`elongated_region_count` than genuinely solid-colored species (*Zebrasoma flavescens*, *Z. scopas*). If they don't - especially if `stripe_present` is near 100% across the board regardless of species - something is wrong with the extraction, not with the fish.

In [ ]:
from collections import defaultdict

by_species = defaultdict(list)
for row in reader:
    by_species[row["image_key"].split("/")[1]].append(row)

check_species = [
    "Acanthurus_lineatus", "Zebrasoma_flavescens", "Zebrasoma_scopas", "Acanthurus_achilles",
]
print(f"{'species':<24} {'n':>3} {'stripe%':>8} {'spot%':>7} {'solid%':>7} {'mean_elong':>11}")
for sp in check_species:
    rows = by_species.get(sp, [])
    if not rows:
        print(f"{sp:<24} not found")
        continue
    n = len(rows)
    stripe_pct = 100 * sum(r["stripe_present"] == "True" for r in rows) / n
    spot_pct = 100 * sum(r["spot_present"] == "True" for r in rows) / n
    solid_pct = 100 * sum(r["is_solid"] == "True" for r in rows) / n
    mean_elong = sum(float(r["elongated_region_count"]) for r in rows) / n
    print(f"{sp:<24} {n:>3} {stripe_pct:>7.0f}% {spot_pct:>6.0f}% {solid_pct:>6.0f}% {mean_elong:>11.1f}")

overall_stripe_pct = 100 * sum(r["stripe_present"] == "True" for r in reader) / len(reader)
print(f"\nOverall stripe_present rate across all {len(reader)} images: {overall_stripe_pct:.0f}%")
print("If Acanthurus_lineatus doesn't score clearly higher than the Zebrasoma species, or the "
      "overall rate is implausibly high (e.g. >70-80%), stop and investigate before trusting "
      "this output for Phase 3.")

## 6b-ii. Which signal is driving stripe_present?

`stripe_present` is True if *either* the region-shape check (elongated_region_count >= min_elongated_region_count) or the FFT periodicity check (periodicity_strength >= min_periodicity_strength) fires - they're meant to be independent, but a bug or an uncalibrated threshold in either one can hide behind the other when only the combined boolean is inspected. This splits the two out per check_species so a future implausible 6b result can be traced to a specific signal instead of guessed at.

In [ ]:
print(f"{'species':<24} {'n':>3} {'region-path%':>13} {'periodicity-path%':>18} "
      f"{'per.mean':>9} {'per.median':>10} {'per.min':>8} {'per.max':>8}")
for sp in check_species:
    rows = by_species.get(sp, [])
    if not rows:
        continue
    n = len(rows)
    elong = [float(r["elongated_region_count"]) for r in rows]
    period = [float(r["periodicity_strength"]) for r in rows]
    region_pct = 100 * sum(e >= config.stripe.min_elongated_region_count for e in elong) / n
    period_pct = 100 * sum(p >= config.stripe.min_periodicity_strength for p in period) / n
    print(f"{sp:<24} {n:>3} {region_pct:>12.1f}% {period_pct:>17.1f}% "
          f"{sum(period)/n:>9.3f} {sorted(period)[n // 2]:>10.3f} {min(period):>8.3f} {max(period):>8.3f}")

## 6c. Visual pattern diagnostics (cluster/region overlay images)

When step 6b's numbers look implausible for a species, staring at aggregate percentages doesn't say *why* - this renders, per inspected image, a 3-panel PNG (original crop / k-means cluster map / flagged-region overlay showing exactly which pixels counted as "stripe-eligible" in red and "spot-eligible" in cyan) and zips them up as a direct download, so you can attach the zip for review any time this stage's output looks off. This is what found the two real bugs fixed in v2.0.5: k-means splitting a genuinely solid-coloured fish into smooth shading zones that scored high eccentricity just from spanning much of the fish's own elongated silhouette, not from being real stripes.

Uses the same `check_species` list as step 6b, and reuses the `config`/`reader`/`by_species` already built above - run 5, 6, and 6b before this.

In [ ]:
import zipfile

import matplotlib.cm as cm
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image
from scipy import ndimage

from pattern_extractor.clustering import assign_clusters, fit_reference

DIAG_OUT_DIR = PROJECT_DIR / "reports" / "pattern_diagnostics"
DIAG_OUT_DIR.mkdir(parents=True, exist_ok=True)


def _find_species_dir(root, species_name):
    matches = list(root.glob(f"*/{species_name}"))
    return matches[0] if matches else None


def _load_image_and_mask(species_dir, stem):
    image = np.array(Image.open(species_dir / f"{stem}.png").convert("RGB"))
    mask = np.array(Image.open(species_dir / f"{stem}_mask.png").convert("L")) > 127
    return image, mask


def _labeled_regions_with_masks(binary_mask, min_area):
    """Mirrors geometry.find_regions()'s math but also returns each region's pixel mask."""
    labeled, n = ndimage.label(binary_mask)
    if n == 0:
        return []
    bounding_boxes = ndimage.find_objects(labeled)
    out = []
    for label_id, bbox in enumerate(bounding_boxes, start=1):
        if bbox is None:
            continue
        component = labeled[bbox] == label_id
        area = int(component.sum())
        if area < min_area:
            continue
        local_rows, local_cols = np.where(component)
        rows = local_rows + bbox[0].start
        cols = local_cols + bbox[1].start
        row_mean, col_mean = rows.mean(), cols.mean()
        dr, dc = rows - row_mean, cols - col_mean
        mu20, mu02, mu11 = np.mean(dr * dr), np.mean(dc * dc), np.mean(dr * dc)
        common = np.sqrt((mu20 - mu02) ** 2 + 4 * mu11**2)
        lambda1 = (mu20 + mu02 + common) / 2
        lambda2 = (mu20 + mu02 - common) / 2
        ratio = lambda2 / lambda1 if lambda1 > 1e-9 else 0.0
        eccentricity = min(float(np.sqrt(max(0.0, 1.0 - ratio))), 1.0)
        full_mask = np.zeros(binary_mask.shape, dtype=bool)
        full_mask[rows, cols] = True
        out.append({"area": area, "eccentricity": eccentricity, "mask": full_mask})
    return out


def _make_diagnostic_figure(image, mask, result, stem, out_dir):
    total_masked = len(result.mask_coords[0])
    min_area = max(1, round(total_masked * config.region.min_region_area_fraction))
    dominant = int(result.fractions.argmax())

    label_img = result.label_image()
    n_k = len(result.fractions)
    cmap = cm.get_cmap("tab10", max(n_k, 1))
    cluster_rgb = np.zeros((*label_img.shape, 3), dtype=np.uint8)
    for c in range(n_k):
        cluster_rgb[label_img == c] = (np.array(cmap(c)[:3]) * 255).astype(np.uint8)
    cluster_rgb[label_img == -1] = (20, 20, 20)

    overlay = (image * 0.3).astype(np.uint8)
    overlay[~mask] = image[~mask]  # leave grey background untouched for context
    stripe_ecc = config.stripe.min_eccentricity_for_stripe
    spot_ecc = config.spot.max_eccentricity_for_spot
    n_stripe_regions = 0
    n_spot_regions = 0
    for cluster_index in range(n_k):
        if cluster_index == dominant:
            continue
        binary = result.binary_mask_for_cluster(cluster_index)
        for region in _labeled_regions_with_masks(binary, min_area):
            if region["eccentricity"] >= stripe_ecc:
                overlay[region["mask"]] = (255, 0, 0)
                n_stripe_regions += 1
            elif region["eccentricity"] <= spot_ecc:
                overlay[region["mask"]] = (0, 220, 255)
                n_spot_regions += 1

    fig, axes = plt.subplots(1, 3, figsize=(16, 5.5))
    axes[0].imshow(image)
    axes[0].set_title(stem, fontsize=9)
    axes[0].axis("off")
    axes[1].imshow(cluster_rgb)
    fracs = ", ".join(f"{f:.0%}" for f in result.fractions)
    axes[1].set_title(f"clusters (dominant=#{dominant})\nfractions: {fracs}", fontsize=9)
    axes[1].axis("off")
    axes[2].imshow(overlay)
    axes[2].set_title(
        f"red=stripe-eligible ({n_stripe_regions})  cyan=spot-eligible ({n_spot_regions})",
        fontsize=9,
    )
    axes[2].axis("off")
    fig.tight_layout()
    out_path = out_dir / f"{stem}.png"
    fig.savefig(out_path, dpi=110)
    plt.close(fig)
    return out_path


saved = []
for species_name in check_species:
    species_dir = _find_species_dir(config.extracted_root, species_name)
    if species_dir is None:
        print(f"skip (not found under {config.extracted_root}): {species_name}")
        continue
    stems = sorted(p.stem for p in species_dir.glob("*.png") if not p.stem.endswith("_mask"))
    ref_stem = config.reference_filename_stem if config.reference_filename_stem in stems else stems[0]
    chosen = stems  # every image for this species, not a sample - the point is full review coverage

    ref_image, ref_mask = _load_image_and_mask(species_dir, ref_stem)
    ref_centers = fit_reference(ref_image, ref_mask, config.clustering)

    species_out = DIAG_OUT_DIR / species_name
    species_out.mkdir(parents=True, exist_ok=True)
    for stem in chosen:
        image, mask = _load_image_and_mask(species_dir, stem)
        result = assign_clusters(image, mask, ref_centers, config.clustering)
        path = _make_diagnostic_figure(image, mask, result, stem, species_out)
        saved.append(path)
        print(f"saved {path}")

zip_path = PROJECT_DIR / "reports" / "pattern_diagnostics.zip"
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for p in saved:
        zf.write(p, arcname=str(p.relative_to(DIAG_OUT_DIR)))

print(f"\n{len(saved)} diagnostic image(s) zipped to {zip_path}")
from google.colab import files
files.download(str(zip_path))

## Next: Phase 3

`reports/pattern_features.csv` is Phase 3's input (per-species aggregation + distance matrices - see README.md's Planned Approach, step 3). It's already saved under Drive; pull it down locally, or keep working from Drive, to continue there.